In [ ]:
#This is testing a random forest model
import pandas as pd
from datascience import *

#Transforms strings into numeric
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

#Define dataset
health_db = pd.read_csv("health_dataset.csv")


In [ ]:
#Transform string values into numeric using label encoding
str_columns = ["student_id", "gender", "part_time_job", "diet_quality", "parental_education_level",
           "internet_quality", "extracurricular_participation"]
for feature in str_columns:
    le = LabelEncoder()
    health_db[feature] = le.fit_transform(health_db[feature])

health_db.info() #Show column values
health_db.head(5) #Show dataset structure


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   student_id                     1000 non-null   int64  
 1   age                            1000 non-null   int64  
 2   gender                         1000 non-null   int64  
 3   study_hours_per_day            1000 non-null   float64
 4   social_media_hours             1000 non-null   float64
 5   netflix_hours                  1000 non-null   float64
 6   part_time_job                  1000 non-null   int64  
 7   attendance_percentage          1000 non-null   float64
 8   sleep_hours                    1000 non-null   float64
 9   diet_quality                   1000 non-null   int64  
 10  exercise_frequency             1000 non-null   int64  
 11  parental_education_level       1000 non-null   int64  
 12  internet_quality               1000 non-null   int64  
 13  

,student_id,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,0,23,0,0.0,1.2,1.1,0,85.0,8.0,0,6,2,0,8,1,56.2
1,1,20,0,6.9,2.8,2.3,0,97.3,4.6,1,6,1,0,8,0,100.0
2,2,21,1,1.4,3.1,1.3,0,94.8,8.0,2,1,1,2,1,0,34.3
3,3,23,0,1.0,3.9,1.0,0,71.0,9.2,2,4,2,1,1,1,26.8
4,4,19,0,5.0,4.4,0.5,0,90.9,4.9,0,3,2,1,1,0,66.4


In [ ]:
#Dataset is not consistent enough for the model. This block will create a function
#that will treat each column as a question that impact the final rating.

#Normalization function
def value_normalize(value, min_value, max_value):
    """Function to normalize numeric values.
    Inputs:
        value: Column referneced
        min_value: Minimum value of column
        max_value: Maximum value of column
    Returns:
        normalize (decimal): Normalized value 
    """
    #Equation for normalization
    normalize = (value - min_value) / (max_value - min_value)

    #Return new value
    #Also caps the value from 0-1. No outliers
    return max(0, min (1, normalize))


#Consistant score calculator function
#More columns can be added in the future if needed
def score_calculator(row):
    """Function to calculate the mental_health_score of every row through normalization. This is to 
    replace inconsistant scoring with calculated scoring based on the same weights.
    Inputs:
        row: Each row in health_db
    Returns:
        final_score: Score based off all normalized columns that will be used
    """
    #Normalizing each column that will be used. row, min, max structure
    sleep_norm = value_normalize(row["sleep_hours"], 4, 8)
    study_norm = value_normalize(row["study_hours_per_day"], 0, 8)
    social_media_norm = value_normalize(row["social_media_hours"], 0, 8)
    tv_norm = value_normalize(row["netflix_hours"], 0, 6)
    job_norm = value_normalize(row["part_time_job"], 0, 1)
    diet_quality_norm = value_normalize(row["diet_quality"], 0, 2)
    exercise_norm = value_normalize(row["exercise_frequency"], 0, 5)
    extra_norm = value_normalize(row["extracurricular_participation"], 0, 1)
    
    #Weight Application to determine final score (Combined they all add to 1.0)
    #Final score equation (w1x1 + ... w?x?)
    final_score = (0.15*exercise_norm + 0.2*sleep_norm + 0.1*study_norm + 
                   0.15*social_media_norm + 0.1*job_norm + 0.1*diet_quality_norm +
                   0.1*tv_norm + 0.1*extra_norm)

    #Return final result
    return final_score

#Drop unused columns (Can be changed later)
health_db

#Replace current ratings with new the ones calculated
health_db['mental_health_rating'] = health_db.apply(score_calculator, axis = 1)

#Save data to new csv
health_db.to_csv("normalized_health_dataset.csv", index = False)
    




In [3]:
#Drops rows that contain missing values under mental_health_rating
health_db = health_db.dropna(subset = ["mental_health_rating"])

#Defining X and y
X = health_db[["study_hours_per_day", "social_media_hours", "netflix_hours", "part_time_job", 
              "sleep_hours", "diet_quality", "exercise_frequency"]]
y = health_db["mental_health_rating"] #Target column

#Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

#Defining model
model = RandomForestClassifier(n_estimators = 100, random_state = 42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [4]:
#Accuracy and model report
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

#Print report and accuracy
print(f"Overall Accuracy: {accuracy:.2f}")
print("-" * 40)
print("\nClassification Report: \n", report)

Overall Accuracy: 0.07
----------------------------------------

Classification Report: 
               precision    recall  f1-score   support

           1       0.00      0.00      0.00        19
           2       0.25      0.18      0.21        22
           3       0.06      0.05      0.05        22
           4       0.06      0.05      0.05        20
           5       0.06      0.04      0.05        25
           6       0.14      0.17      0.16        23
           7       0.00      0.00      0.00        13
           8       0.06      0.11      0.08        18
           9       0.00      0.00      0.00        19
          10       0.05      0.05      0.05        19

    accuracy                           0.07       200
   macro avg       0.07      0.07      0.07       200
weighted avg       0.07      0.07      0.07       200

